<div style='background:linear-gradient(135deg,#1A2E4A 0%,#0D7377 100%);padding:50px 40px;border-radius:12px;color:white;text-align:center;font-family:Arial,sans-serif;'>
  <p style='font-size:13px;letter-spacing:3px;color:#14BDBD;margin:0 0 8px 0;'>AI / ML FOUNDATIONS COHORT</p>
  <h1 style='font-size:40px;margin:0 0 8px 0;font-weight:900;'>MEETING 7</h1>
  <h2 style='font-size:24px;font-weight:300;margin:0 0 30px 0;color:#D0D7E3;'>Unsupervised Learning</h2>
  <div style='width:60px;height:3px;background:#F0A500;margin:0 auto 30px auto;'></div>
  <p style='font-size:14px;color:#D0D7E3;margin:0 0 6px 0;'>K-Means · DBSCAN · Hierarchical · PCA · t-SNE · Anomaly Detection</p>
  <p style='font-size:13px;color:#6B8A9A;margin:0;'>Open in Jupyter or Google Colab</p>
</div>


---
## 🗺️ Session Roadmap

| Part | Topic | Time |
|------|-------|------|
| A | What is Unsupervised Learning? | 10 min |
| B | K-Means Clustering | 25 min |
| C | DBSCAN & Hierarchical Clustering | 20 min |
| D | Interpreting & Profiling Clusters | 15 min |
| E | PCA — Dimensionality Reduction | 20 min |
| F | t-SNE — Visualising High-Dimensional Data | 15 min |
| G | Anomaly Detection | 15 min |
| H | Mini Tasks + Project | 20 min |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,
                      'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.25})

# ── Main Dataset: E-commerce Customer Behaviour ──────────────────────────────
from sklearn.preprocessing import StandardScaler

n = 500
np.random.seed(42)
# 4 natural customer segments
segments = [
    # recency, frequency, monetary, session_len, support_calls
    dict(recency=(5,2),  frequency=(20,4), monetary=(800,150),  session=(45,10), support=(1,1)),
    dict(recency=(30,8), frequency=(5,2),  monetary=(150,50),   session=(12,5),  support=(4,2)),
    dict(recency=(15,5), frequency=(12,3), monetary=(400,100),  session=(30,8),  support=(2,1)),
    dict(recency=(60,15),frequency=(2,1),  monetary=(50,20),    session=(5,3),   support=(7,3)),
]
sizes = [150, 120, 140, 90]

rows = []
for seg, size in zip(segments, sizes):
    rows.append(np.column_stack([
        np.abs(np.random.normal(seg['recency'][0],   seg['recency'][1],   size)),
        np.abs(np.random.normal(seg['frequency'][0], seg['frequency'][1], size)),
        np.abs(np.random.normal(seg['monetary'][0],  seg['monetary'][1],  size)),
        np.abs(np.random.normal(seg['session'][0],   seg['session'][1],   size)),
        np.abs(np.random.normal(seg['support'][0],   seg['support'][1],   size)),
    ]))

X_raw = np.vstack(rows)
true_labels = np.repeat([0,1,2,3], sizes)
feature_names = ['recency_days','purchase_freq','total_spend','avg_session_min','support_calls']

df = pd.DataFrame(X_raw, columns=feature_names)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f'Dataset: {df.shape[0]} customers × {df.shape[1]} features')
print(f'No labels — we will discover segments ourselves')
df.describe().round(2)


---

# 🔍 Part A: What is Unsupervised Learning?
#### *Finding structure where no labels exist*

---

In supervised learning, you have labels — correct answers for every training sample.  
In **unsupervised learning**, there are no labels. The algorithm discovers structure itself.

### Three Core Tasks
| Task | Question | Algorithms |
|------|---------|------------|
| **Clustering** | Which samples naturally group together? | K-Means, DBSCAN, Hierarchical |
| **Dimensionality Reduction** | How can we compress data with minimal info loss? | PCA, t-SNE, UMAP |
| **Anomaly Detection** | Which points do not belong? | Isolation Forest, LOF, Autoencoders |

### Real-World Uses
- 🛒 **Customer segmentation** — group buyers by behaviour without predefined categories
- 🔐 **Fraud detection** — flag transactions that don't match normal patterns
- 🧬 **Gene expression analysis** — discover natural disease subtypes
- 📰 **Topic modelling** — discover themes in millions of documents automatically
- 🖼️ **Image compression** — PCA reduces image dimensions while preserving visual content
- 📉 **Feature engineering** — PCA components as inputs to downstream supervised models

> ⚠️ **Unsupervised results are harder to evaluate** — there is no ground truth.  
> You evaluate using domain knowledge, internal metrics, and business interpretability.


---

# ⭕ Part B: K-Means Clustering
#### *Partition data into K groups by minimising within-cluster variance*

---

### The Algorithm
1. **Initialise** K centroids (randomly or with K-Means++ smart seeding)
2. **Assign** each point to its nearest centroid
3. **Update** each centroid to the mean of its assigned points
4. **Repeat** until assignments stop changing

$$J = \sum_{k=1}^K \sum_{x_i \in C_k} \|x_i - \mu_k\|^2 \quad \text{(Inertia — minimise this)}$$

### What K-Means Assumes
- Clusters are **spherical** (roughly equal spread in all directions)
- Clusters are **similar in size**
- The number of clusters **K is known** in advance

### When These Assumptions Break Down
- Elongated clusters → use DBSCAN or Gaussian Mixture Models
- Very different sized clusters → use DBSCAN
- Unknown K → use Elbow method, Silhouette score, or Gap statistic


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

# ── Step-by-step K-Means iteration visualised ────────────────────────────────
# Use 2D slice for visualisation
X_2d = X[:, :2]   # recency and frequency (scaled)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['#0D7377','#F0A500','#E74C3C','#8E44AD']

np.random.seed(1)
centroids = X_2d[np.random.choice(len(X_2d), 4, replace=False)]

for step in range(4):
    # Assign
    dists  = np.linalg.norm(X_2d[:,None,:] - centroids[None,:,:], axis=2)
    labels = np.argmin(dists, axis=1)
    axes[step].scatter(X_2d[:,0], X_2d[:,1], c=[colors[l] for l in labels],
                       alpha=0.4, s=20)
    axes[step].scatter(centroids[:,0], centroids[:,1], c=colors[:4],
                       s=200, marker='*', edgecolors='black', linewidths=1.5, zorder=5)
    axes[step].set_title(f'Iteration {step+1}', fontweight='bold')
    axes[step].set_xlabel('Recency (scaled)'); axes[step].set_ylabel('Frequency (scaled)')
    # Update centroids
    centroids = np.array([X_2d[labels==k].mean(0) for k in range(4)])

plt.suptitle('K-Means: Centroid Convergence Over Iterations', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── Choosing K: Elbow Method + Silhouette Score ──────────────────────────────
K_range     = range(2, 11)
inertias    = []
silhouettes = []

for k in K_range:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, lbl))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(K_range, inertias, 'o-', color='#0D7377', linewidth=2, markersize=7)
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia (within-cluster variance)')
axes[0].set_title('Elbow Method — look for the bend', fontweight='bold')

axes[1].plot(K_range, silhouettes, 's-', color='#F0A500', linewidth=2, markersize=7)
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score — higher is better (max=1)', fontweight='bold')
best_k = K_range[np.argmax(silhouettes)]
axes[1].axvline(best_k, color='red', linestyle='--', label=f'Best K={best_k}')
axes[1].legend()

plt.tight_layout(); plt.show()
print(f'Best K by Silhouette: {best_k}')
print(f'Silhouette score at K={best_k}: {max(silhouettes):.4f}')
print('Silhouette ranges: [-1, 1]. Above 0.5 = reasonable. Above 0.7 = strong clusters.')


In [ ]:
# ── Silhouette plot — per-sample analysis ─────────────────────────────────────
from sklearn.metrics import silhouette_samples

km_best    = KMeans(n_clusters=4, random_state=42, n_init=10)
km_labels  = km_best.fit_predict(X)
sil_vals   = silhouette_samples(X, km_labels)

fig, ax = plt.subplots(figsize=(9, 5))
y_lower = 10
palette = ['#0D7377','#F0A500','#E74C3C','#8E44AD']

for k in range(4):
    cluster_sil = np.sort(sil_vals[km_labels == k])
    size_k      = len(cluster_sil)
    y_upper     = y_lower + size_k
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil,
                     facecolor=palette[k], edgecolor=palette[k], alpha=0.8)
    ax.text(-0.05, y_lower + 0.5*size_k, f'C{k}', fontsize=10, fontweight='bold')
    y_lower = y_upper + 10

mean_sil = sil_vals.mean()
ax.axvline(mean_sil, color='red', linestyle='--', label=f'Mean = {mean_sil:.3f}')
ax.set_xlabel('Silhouette coefficient'); ax.set_ylabel('Cluster samples')
ax.set_title('Silhouette Plot — Quality of Each Cluster', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()
print('Thin/negative slices = poorly assigned points — consider adjusting K.')


---

# 🌐 Part C: DBSCAN & Hierarchical Clustering
#### *When K-Means assumptions do not hold*

---

### DBSCAN — Density-Based Spatial Clustering
DBSCAN defines clusters as **dense regions** separated by sparse regions.  
It does not require K — it finds clusters automatically.

**Two parameters:**
- **eps (ε)** — the neighbourhood radius (how close must points be?)
- **min_samples** — minimum points within eps to be a core point

**Three point types:**
| Type | Definition |
|------|------------|
| **Core point** | Has ≥ min_samples within eps — anchor of a cluster |
| **Border point** | Within eps of a core point but not a core itself |
| **Noise point** | Not within eps of any core point — labelled -1 |

### DBSCAN Advantages Over K-Means
- Finds **arbitrary shaped** clusters (not just spherical)
- **Automatically detects noise/outliers** (labelled -1)
- Does **not require K** to be specified in advance
- Can find **any number of clusters**


In [ ]:
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from sklearn.datasets import make_moons, make_blobs

# ── K-Means vs DBSCAN on non-spherical data ──────────────────────────────────
X_moon, _ = make_moons(n_samples=300, noise=0.08, random_state=42)
X_blob, _ = make_blobs(n_samples=300, centers=3, random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

datasets = [('Moons (non-spherical)', X_moon), ('Blobs (spherical)', X_blob)]
for col, (name, Xd) in enumerate(datasets):
    # K-Means
    km_lbl = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(Xd)
    axes[0][col].scatter(Xd[:,0], Xd[:,1], c=km_lbl, cmap='Set1', alpha=0.7, s=25)
    axes[0][col].set_title(f'K-Means on {name}', fontweight='bold')

    # DBSCAN
    eps_val = 0.3 if 'Moon' in name else 1.0
    db_lbl  = DBSCAN(eps=eps_val, min_samples=8).fit_predict(Xd)
    n_cls   = len(set(db_lbl)) - (1 if -1 in db_lbl else 0)
    n_noise = (db_lbl==-1).sum()
    colors  = ['red' if l==-1 else cm.Set1(l/max(1,n_cls)) for l in db_lbl]
    axes[1][col].scatter(Xd[:,0], Xd[:,1], c=colors, alpha=0.7, s=25)
    axes[1][col].set_title(f'DBSCAN on {name}\n({n_cls} clusters, {n_noise} noise)',
                            fontweight='bold')

axes[0][0].set_ylabel('K-Means', fontsize=11, fontweight='bold')
axes[1][0].set_ylabel('DBSCAN', fontsize=11, fontweight='bold')
plt.suptitle('K-Means vs DBSCAN — When Shape Matters', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print('K-Means fails on moons (wrong shapes). DBSCAN handles them perfectly.')


In [ ]:
# ── DBSCAN on our customer data: finding outlier customers ────────────────────
db = DBSCAN(eps=0.8, min_samples=10)
db_labels = db.fit_predict(X)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = (db_labels == -1).sum()

print(f'DBSCAN Results:')
print(f'  Clusters found:  {n_clusters}')
print(f'  Noise points:    {n_noise}  ({n_noise/len(db_labels):.1%} of customers)')

# Profile noise points — are they unusual?
noise_mask  = db_labels == -1
normal_mask = ~noise_mask
print(f'\nNoise vs Normal Customer Profile:')
print(f'  {"Feature":<22} {"Normal":>12} {"Noise/Outlier":>16}')
print('  '+'-'*52)
for feat in feature_names:
    nm = df.loc[normal_mask, feat].mean()
    om = df.loc[noise_mask,  feat].mean()
    print(f'  {feat:<22} {nm:>12.1f} {om:>16.1f}')


In [ ]:
# ── Hierarchical Clustering — Dendrograms ─────────────────────────────────────
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

# Use a small sample for readable dendrogram
sample_idx = np.random.choice(len(X), 60, replace=False)
X_sample   = X[sample_idx]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
linkage_methods = [('ward','Ward'), ('complete','Complete'), ('average','Average')]

for ax, (method, title) in zip(axes, linkage_methods):
    Z = linkage(X_sample, method=method)
    dendrogram(Z, ax=ax, leaf_rotation=90, leaf_font_size=0,
               color_threshold=0.7*max(Z[:,2]))
    ax.set_title(f'Linkage: {title}', fontweight='bold')
    ax.set_xlabel('Sample index'); ax.set_ylabel('Distance')

plt.suptitle('Hierarchical Clustering Dendrograms\n(Cut the tree horizontally to choose K)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# Cut the tree to get 4 clusters
Z_full  = linkage(X, method='ward')
hc_labels = fcluster(Z_full, t=4, criterion='maxclust') - 1
print(f'Hierarchical clustering (Ward, 4 clusters):')
print(pd.Series(hc_labels).value_counts().sort_index().to_string())


---

# 📋 Part D: Interpreting & Profiling Clusters
#### *Turning cluster IDs into business insight*

---

Finding clusters is the easy part. **Understanding what they mean** is the real work.

A cluster ID (0, 1, 2, 3) has no intrinsic meaning — you must characterise each cluster  
using its feature distributions and give it a business interpretation.

### The Cluster Profiling Process
1. Compute mean (and std) of each feature per cluster
2. Compare against the overall mean — which features distinguish each cluster?
3. Identify the defining characteristic of each cluster
4. Give each cluster a business label
5. Report: size, key metrics, recommended actions


In [ ]:
# ── Full cluster profiling ────────────────────────────────────────────────────
df_profiled = df.copy()
df_profiled['cluster'] = km_labels

profile = df_profiled.groupby('cluster')[feature_names].mean().round(1)
sizes   = df_profiled['cluster'].value_counts().sort_index()

# Z-score profile — how far each cluster deviates from the overall mean
overall_mean = df[feature_names].mean()
overall_std  = df[feature_names].std()
z_profile    = (profile - overall_mean) / overall_std

print('=== Raw Feature Means Per Cluster ===')
print(profile.to_string())
print(f'\n=== Cluster Sizes ===')
for k, sz in sizes.items():
    print(f'  Cluster {k}: {sz} customers ({sz/len(df):.0%})')

# Heatmap of z-score profiles
fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(z_profile, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax,
            xticklabels=[f.replace('_',' ') for f in feature_names])
ax.set_title('Cluster Profile Heatmap (Z-scores from overall mean)',
             fontweight='bold')
ax.set_ylabel('Cluster')
plt.tight_layout(); plt.show()

print('\nGreen = above average  |  Red = below average')
print('Use this heatmap to name and characterise each cluster.')


In [ ]:
# ── Give clusters business names and recommendations ─────────────────────────
cluster_labels = {
    0: ('Champions', 'Low recency, high freq, high spend', 'Reward & retain'),
    1: ('At Risk',   'High recency, low freq, low spend',  'Win-back campaign'),
    2: ('Loyal',     'Mid recency, good freq, mid spend',  'Upsell opportunities'),
    3: ('Lost',      'Very high recency, low everything',  'Final reactivation or remove'),
}
# Note: your actual cluster numbers may differ — reassign based on your profiling

print('=== Business Segment Report ===')
print(f'  {"#":<3} {"Segment":<15} {"Profile":<38} {"Action":<30} {"Size":>6}')
print('  '+'-'*97)
for k, (name, profile_str, action) in cluster_labels.items():
    sz = sizes.get(k, 0)
    print(f'  {k:<3} {name:<15} {profile_str:<38} {action:<30} {sz:>6}')

# Radar chart for one cluster
from matplotlib.patches import FancyArrowPatch
angles = np.linspace(0, 2*np.pi, len(feature_names), endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(1, 4, figsize=(16, 4), subplot_kw=dict(polar=True))
palette = ['#0D7377','#F0A500','#E74C3C','#8E44AD']

z_norm = (z_profile - z_profile.min()) / (z_profile.max() - z_profile.min() + 1e-8)

for k, ax in enumerate(axes):
    vals = z_norm.iloc[k].tolist() + [z_norm.iloc[k].tolist()[0]]
    ax.plot(angles, vals, color=palette[k], linewidth=2)
    ax.fill(angles, vals, color=palette[k], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([f.replace('_','\n') for f in feature_names], fontsize=7)
    name = cluster_labels.get(k, (f'Cluster {k}',))[0]
    ax.set_title(f'Cluster {k}\n{name}', fontweight='bold', fontsize=9, pad=15)
    ax.set_ylim(0, 1)

plt.suptitle('Customer Segment Radar Profiles', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


---

## ⏱️ Mini Task 1 — *Cluster & Profile Your Segments*

> **12 minutes** — solo or pairs. Run, interpret, share.

---


Using the customer dataset:

1. Run K-Means with K=3 and K=5 — compare silhouette scores
2. Choose the better K and profile all clusters using the heatmap approach
3. Give each cluster a business name based on its profile
4. Which cluster would you prioritise for a retention campaign? Why?


In [ ]:
# ✏️  YOUR TURN

# # K=3 vs K=5
# for k in [3, 5]:
#     km  = KMeans(n_clusters=k, random_state=42, n_init=10)
#     lbl = km.fit_predict(X)
#     print(f'K={k} Silhouette: {silhouette_score(X, lbl):.4f}')
# 
# # Profile the better one
# # Add cluster to df_profiled
# # Plot heatmap
# # Name your clusters in a comment

# ── Write below ──────────────────────────────────────────────


---

# 🔭 Part E: PCA — Principal Component Analysis
#### *Compress information, reveal structure*

---

PCA finds the **directions of maximum variance** in your data — called principal components —  
and projects the data onto them. The result is a new coordinate system where:

- **PC1** captures the most variance in the data
- **PC2** captures the second most, and is orthogonal (perpendicular) to PC1
- And so on...

$$\text{PCA solves: } \text{find } \vec{v} \text{ that maximises } \text{Var}(X\vec{v}) \text{ subject to } \|\vec{v}\| = 1$$

### Three Main Uses in ML
| Use | How |
|-----|-----|
| **Visualisation** | Reduce to 2D or 3D to plot high-dimensional data |
| **Noise reduction** | Drop components that capture mostly noise |
| **Preprocessing** | Use components as inputs to downstream models |

> ⚠️ **Always standardise features before PCA** — otherwise features with larger scales  
> will dominate the principal components, regardless of their actual importance.


In [ ]:
from sklearn.decomposition import PCA

# ── Full PCA analysis ─────────────────────────────────────────────────────────
pca = PCA()
pca.fit(X)   # X is already standardised

explained   = pca.explained_variance_ratio_ * 100
cumulative  = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(1, len(explained)+1), explained, color='#0D7377', alpha=0.8)
axes[0].plot(range(1, len(explained)+1), cumulative, 'o-', color='#F0A500',
             linewidth=2, label='Cumulative')
axes[0].axhline(90, color='red', linestyle='--', label='90% threshold')
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('Scree Plot — Explained Variance per Component', fontweight='bold')
axes[0].legend()

# Loadings heatmap — what does each component represent?
loadings = pd.DataFrame(pca.components_[:3].T, index=feature_names,
                         columns=['PC1','PC2','PC3'])
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=axes[1], linewidths=0.5)
axes[1].set_title('PCA Loadings — What Each Component Captures', fontweight='bold')

plt.tight_layout(); plt.show()

n_90 = np.argmax(cumulative >= 90) + 1
print(f'Components needed for 90% variance: {n_90} out of {X.shape[1]}')
print(f'\nPC1 ({explained[0]:.1f}%) captures:')
top_pc1 = loadings['PC1'].abs().sort_values(ascending=False)
for feat, loading in top_pc1.items():
    sign = '+' if loadings.loc[feat,'PC1'] > 0 else '-'
    print(f'  {sign}{feat}: {abs(loading):.3f}')


In [ ]:
# ── PCA for visualisation — plot clusters in 2D ──────────────────────────────
pca_2d = PCA(n_components=2)
X_2d_pca = pca_2d.fit_transform(X)

pca_3d = PCA(n_components=3)
X_3d_pca = pca_3d.fit_transform(X)

fig = plt.figure(figsize=(14, 5))
palette = ['#0D7377','#F0A500','#E74C3C','#8E44AD']

# 2D PCA with K-Means labels
ax1 = fig.add_subplot(121)
for k in range(4):
    mask = km_labels == k
    ax1.scatter(X_2d_pca[mask,0], X_2d_pca[mask,1], color=palette[k],
                label=f'Cluster {k}', alpha=0.7, s=30)
ax1.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} var)')
ax1.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} var)')
ax1.set_title('K-Means Clusters in PCA Space (2D)', fontweight='bold')
ax1.legend()

# 3D PCA
ax2 = fig.add_subplot(122, projection='3d')
for k in range(4):
    mask = km_labels == k
    ax2.scatter(X_3d_pca[mask,0], X_3d_pca[mask,1], X_3d_pca[mask,2],
                color=palette[k], label=f'C{k}', alpha=0.5, s=20)
ax2.set_xlabel('PC1'); ax2.set_ylabel('PC2'); ax2.set_zlabel('PC3')
ax2.set_title('Clusters in 3D PCA Space', fontweight='bold')
ax2.legend(fontsize=8)

plt.tight_layout(); plt.show()
var_2d = pca_2d.explained_variance_ratio_.sum()
var_3d = pca_3d.explained_variance_ratio_.sum()
print(f'2D PCA retains {var_2d:.1%} of original variance')
print(f'3D PCA retains {var_3d:.1%} of original variance')


In [ ]:
# ✏️  YOUR TURN

# 1. Use PCA to reduce X to the minimum components needed for 95% variance
# 2. Train a KMeans model on the PCA-reduced data
# 3. Compare silhouette score to clustering on the full X
# 4. Plot a biplot: scatter of PC1 vs PC2 with feature loading arrows
#    Hint: arrows go at pca.components_.T[:,0] and pca.components_.T[:,1]

# ── Write below ──────────────────────────────────────────────


---

# 🗺️ Part F: t-SNE — Visualising High-Dimensional Data
#### *Non-linear dimensionality reduction for exploration*

---

t-SNE (t-distributed Stochastic Neighbour Embedding) is a **non-linear** dimensionality  
reduction technique specifically designed for **visualisation**.

It preserves **local structure** — points that are similar in the original space  
will be close together in the 2D plot. It is not suitable for general compression.

### t-SNE vs PCA
| Aspect | PCA | t-SNE |
|--------|-----|-------|
| Type | Linear | Non-linear |
| Purpose | Compression + visualisation | Visualisation only |
| Speed | Very fast | Slow on large data |
| Reproducible | Yes | Depends on random seed |
| Preserves | Global structure | Local structure |
| Can be reversed | Yes (approximately) | No |

### Key Hyperparameter: Perplexity
Perplexity controls the effective number of neighbours each point considers.  
- Low perplexity (5–15): very local — tighter, more separated clusters
- High perplexity (50+): more global — smoother layout
- Rule of thumb: perplexity between 5 and 50, use 30 as default


In [ ]:
from sklearn.manifold import TSNE

# ── t-SNE with different perplexity values ────────────────────────────────────
perplexities = [5, 15, 30, 50]
fig, axes    = plt.subplots(1, 4, figsize=(16, 4))
palette      = ['#0D7377','#F0A500','#E74C3C','#8E44AD']

for ax, perp in zip(axes, perplexities):
    tsne    = TSNE(n_components=2, perplexity=perp, random_state=42, n_iter=500)
    X_tsne  = tsne.fit_transform(X)
    for k in range(4):
        mask = km_labels == k
        ax.scatter(X_tsne[mask,0], X_tsne[mask,1],
                   color=palette[k], label=f'C{k}', alpha=0.7, s=20)
    ax.set_title(f'Perplexity={perp}', fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

axes[0].legend(fontsize=8)
plt.suptitle('t-SNE Visualisation at Different Perplexity Values', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print('Different perplexity = different layouts. Run multiple and compare.')
print('The clusters we found with K-Means should appear as separated blobs here.')


In [ ]:
# ── PCA → t-SNE: best practice pipeline for high-dimensional data ─────────────
# For data with many features: first reduce to ~50 dims with PCA, then apply t-SNE
# This removes noise and dramatically speeds up t-SNE

from sklearn.datasets import load_digits

# Real dataset: 1797 handwritten digits, 64 features each
digits  = load_digits()
X_dig   = digits.data
y_dig   = digits.target

# Step 1: PCA to 30 dims
pca_pre = PCA(n_components=30, random_state=42)
X_pca   = pca_pre.fit_transform(X_dig)
print(f'Original: {X_dig.shape} → After PCA: {X_pca.shape}')
print(f'PCA retains: {pca_pre.explained_variance_ratio_.sum():.1%}')

# Step 2: t-SNE to 2D
tsne_final = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_viz      = tsne_final.fit_transform(X_pca)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_viz[:,0], X_viz[:,1], c=y_dig, cmap='tab10',
                       alpha=0.7, s=20)
plt.colorbar(scatter, label='Digit class')
plt.title('t-SNE of Handwritten Digits (PCA→t-SNE pipeline)\n'
          '64 features → 30 PCA → 2D t-SNE', fontweight='bold')
plt.xticks([]); plt.yticks([])
plt.tight_layout(); plt.show()
print('Each colour = a digit (0-9). No labels used in the visualisation.')
print('Well-separated blobs = the model can likely classify these well.')


---

# 🚨 Part G: Anomaly Detection
#### *Finding what does not belong*

---

Anomaly detection identifies data points that are **significantly different** from  
the rest of the distribution. In ML, anomalies can represent:

- 💳 Fraudulent transactions
- 🏭 Faulty manufacturing parts
- 🏥 Rare disease patterns
- 🔐 Intrusion in network traffic
- 📦 Data entry errors in datasets

### Three Approaches
| Method | How | Best For |
|--------|-----|----------|
| **Isolation Forest** | Anomalies are isolated faster by random splits | General purpose |
| **Local Outlier Factor (LOF)** | Compares point density to neighbours | Local anomalies |
| **One-Class SVM** | Learns a tight boundary around normal data | High-dimensional |


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

# ── Generate data with anomalies ─────────────────────────────────────────────
np.random.seed(42)
n_normal  = 400
n_anomaly = 20

# Normal transactions: moderate amounts, regular frequency
normal = np.column_stack([
    np.random.normal(500, 150, n_normal),    # transaction amount
    np.random.normal(3.0, 0.8, n_normal),    # transactions per day
    np.random.normal(10,  3,   n_normal),    # account age months
])

# Anomalies: unusual patterns
anomalies = np.column_stack([
    np.random.uniform(3000, 8000, n_anomaly),   # very high amounts
    np.random.uniform(15,   30,   n_anomaly),   # very high frequency
    np.random.uniform(0,    2,    n_anomaly),   # very new accounts
])

X_all   = np.vstack([normal, anomalies])
y_true  = np.array([1]*n_normal + [-1]*n_anomaly)  # 1=normal, -1=anomaly

# ── Three detectors ───────────────────────────────────────────────────────────
detectors = [
    ('Isolation Forest', IsolationForest(contamination=0.05, random_state=42)),
    ('Local Outlier Factor', LocalOutlierFactor(n_neighbors=20, contamination=0.05)),
    ('One-Class SVM', OneClassSVM(kernel='rbf', nu=0.05)),
]

X_scaled_ad = StandardScaler().fit_transform(X_all)
print(f'  {"Detector":<25} {"Precision":>12} {"Recall":>10} {"Anomalies Found":>18}')
print('  '+'-'*67)

from sklearn.metrics import precision_score, recall_score
all_preds = {}
for name, det in detectors:
    if hasattr(det, 'fit_predict'):
        pred = det.fit_predict(X_scaled_ad)
    else:
        det.fit(X_scaled_ad[y_true==1])   # One-class SVM trained on normal only
        pred = det.predict(X_scaled_ad)
    all_preds[name] = pred
    prec = precision_score(y_true, pred, pos_label=-1)
    rec  = recall_score(y_true, pred, pos_label=-1)
    n_found = (pred==-1).sum()
    print(f'  {name:<25} {prec:>12.2%} {rec:>10.2%} {n_found:>18}')


In [ ]:
# ── Visualise anomaly detection ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, pred) in zip(axes, all_preds.items()):
    normal_mask  = pred == 1
    anomaly_mask = pred == -1
    ax.scatter(X_all[normal_mask, 0], X_all[normal_mask, 1],
               color='#0D7377', alpha=0.5, s=20, label='Normal')
    ax.scatter(X_all[anomaly_mask, 0], X_all[anomaly_mask, 1],
               color='red', s=80, label='Anomaly', zorder=5, marker='X')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Transaction Amount ($)')
    ax.set_ylabel('Transactions/Day')
    ax.legend(fontsize=8)

plt.suptitle('Anomaly Detection: Three Methods Compared', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# Anomaly score from Isolation Forest (lower = more anomalous)
iso = IsolationForest(contamination=0.05, random_state=42)
iso.fit(X_scaled_ad)
scores = iso.score_samples(X_scaled_ad)

plt.figure(figsize=(9, 3))
plt.hist(scores[y_true==1],  bins=40, alpha=0.7, color='#0D7377', label='Normal')
plt.hist(scores[y_true==-1], bins=10, alpha=0.7, color='red',     label='True anomaly')
plt.axvline(iso.threshold_, color='black', linestyle='--', label='Decision threshold')
plt.xlabel('Anomaly Score (lower = more anomalous)')
plt.title('Isolation Forest Score Distribution', fontweight='bold')
plt.legend(); plt.tight_layout(); plt.show()


---

## ⏱️ Mini Task 2 — *Full Unsupervised Pipeline*

> **15 minutes** — solo or pairs. Run, interpret, share.

---


Apply the **full unsupervised workflow** to the customer dataset:

1. Run K-Means and DBSCAN — compare results
2. Use PCA to visualise the clusters in 2D — do the clusters separate clearly?
3. Identify any anomalous customers using Isolation Forest
4. Build a cluster profile table with business segment names
5. Which segment would you target first for a loyalty programme?


In [ ]:
# ✏️  YOUR TURN

# # Step 1: K-Means and DBSCAN on X
# 
# # Step 2: PCA 2D visualisation with cluster colours
# 
# # Step 3: Isolation Forest on X
# iso = IsolationForest(contamination=0.05, random_state=42)
# anomaly_labels = iso.fit_predict(X)
# print(f'Anomalies: {(anomaly_labels==-1).sum()}')
# 
# # Step 4: Profile table — groupby cluster, mean of all features
# 
# # Step 5: Your segment priority recommendation (as a comment)

# ── Write below ──────────────────────────────────────────────


---
## ✅ What You Covered in Meeting 7
| Concept | Depth |
|---------|-------|
| K-Means — algorithm, inertia, elbow, silhouette, radar profiles | Deep |
| DBSCAN — density clustering, noise points, eps/min_samples | Deep |
| Hierarchical clustering — dendrograms, linkage methods | Solid |
| Cluster profiling — z-scores, heatmaps, business naming | Deep |
| PCA — scree plot, loadings, explained variance, 2D/3D visualisation | Deep |
| t-SNE — perplexity, PCA→t-SNE pipeline, digits visualisation | Deep |
| Anomaly detection — Isolation Forest, LOF, One-Class SVM | Deep |

> 🔭 **Next — Neural Networks:** perceptrons, deep networks, backprop in PyTorch and TensorFlow.

<div style='background:linear-gradient(135deg,#1A2E4A 0%,#0D7377 100%);padding:28px;border-radius:10px;color:white;text-align:center;'>
  <h3 style='margin:0 0 8px 0;color:#14BDBD;'>You can now find patterns no one labelled for you.</h3>
  <p style='color:#F0A500;font-weight:bold;margin:0;'>See you in the Neural Networks session. 🚀</p>
</div>
